# Brent Oil Price — Exploratory Data Analysis & Change Point Modeling
**Birhan Energies | Week 10 Interim Submission**

This notebook covers:
1. Data loading and cleaning
2. Time series properties (trend, stationarity, volatility)
3. Visual EDA with geopolitical/OPEC event overlays
4. Explanation of the Bayesian change point modeling approach used in Task 2


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller

df = pd.read_csv("../data/processed/brent_clean.csv", parse_dates=["Date"])
df.head()


## 2. Event Dataset

We load a structured table of 16 major geopolitical, OPEC policy, and macroeconomic
events that are plausibly relevant to Brent oil price movements over 1987–2022.


In [ ]:
events = pd.read_csv("../data/events.csv", parse_dates=["date"])
events


## 3. Price History with Event Overlays

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))
ax.plot(df.Date, df.Price, color="#2980b9", linewidth=0.8, label="Brent Oil Price")
for _, row in events.iterrows():
    ax.axvline(row["date"], color="red", alpha=0.4, linewidth=0.8)
    ax.text(row["date"], df.Price.max() * 0.95, row["event"], rotation=90, fontsize=6, color="darkred", va="top")
ax.set_title("Brent Oil Prices 1987-2022 with Key Events", fontweight="bold", fontsize=13)
ax.set_ylabel("Price (USD/barrel)")
ax.set_xlabel("Date")
ax.legend()
plt.tight_layout()
plt.show()


## 4. Time Series Properties

### 4.1 Trend
The raw price series shows clear long-run trends and regime shifts (e.g., the 1998 low,
the 2008 spike, the 2014-2016 collapse, the 2020 COVID crash) rather than a stable
long-run mean — a first sign of non-stationarity.

### 4.2 Stationarity (Augmented Dickey-Fuller Test)


In [ ]:
for col, name in [("Price", "Raw Price"), ("log_return", "Log Returns")]:
    result = adfuller(df[col].dropna())
    print(f"{name}: ADF={result[0]:.4f}, p={result[1]:.4f}, Stationary={result[1] < 0.05}")


**Interpretation:** The raw price series fails to reject the unit-root null hypothesis
(p > 0.05) and is therefore **non-stationary** — consistent with the visible long-run
trends and regime shifts above. Log returns strongly reject the null (p < 0.001) and are
**stationary**, which is why log returns (not raw price) are used as the input series for
the Bayesian change point model.


### 4.3 Volatility — Rolling Mean & Standard Deviation

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8))
roll = df.set_index("Date")["Price"].rolling(90)
axes[0].plot(df.Date, df.Price, color="#95a5a6", linewidth=0.5, label="Price")
axes[0].plot(df.Date, roll.mean().values, color="#2980b9", linewidth=1.5, label="90-day Rolling Mean")
axes[0].set_title("Rolling Mean", fontweight="bold")
axes[0].legend()
axes[1].plot(df.Date, roll.std().values, color="#e67e22", linewidth=1)
axes[1].set_title("Rolling Standard Deviation (Volatility)", fontweight="bold")
plt.tight_layout()
plt.show()


**Interpretation:** Volatility is clearly time-varying rather than constant — clustering
around known crisis periods (2008-09, 2014-15, 2020). This volatility clustering is a
key motivation for using a change point model rather than assuming a single global
distribution for the entire 35-year period.


### 4.4 Log Return Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hist(df.Price, bins=50, color="#3498db", edgecolor="white")
axes[0].set_title("Price Distribution", fontweight="bold")
axes[1].hist(df.log_return, bins=100, color="#e74c3c", edgecolor="white")
axes[1].set_title("Log Return Distribution", fontweight="bold")
plt.tight_layout()
plt.show()


## 5. Change Point Models: Purpose and Expected Outputs

**Purpose.** A Bayesian change point model is used to detect one or more points in time
(`tau`) at which the statistical properties of a time series — here, the mean and/or
volatility of Brent log returns — shift abruptly. Rather than assuming the series is
generated by a single stationary process throughout, the model treats it as potentially
composed of distinct regimes separated by structural breaks, and estimates the most
probable location(s) of those breaks directly from the data.

**Approach used (Task 2).** We fit a PyMC-based Bayesian model where the change point
location `tau` and the before/after regime parameters (mean `mu`, volatility `sigma`) are
treated as random variables with priors, and their posterior distributions are estimated
via MCMC sampling.

**Expected outputs:**
- A posterior distribution over the change point date `tau`, from which a most-probable
  change point date is derived.
- Posterior estimates of the mean and volatility of log returns *before* and *after* the
  change point, quantifying the magnitude of the regime shift.
- A qualitative match between the detected change point and the nearest entries in the
  event dataset, used to *contextualize* — not causally explain — the statistical break.

**Inherent limitations:**
- Detects statistical association in time, not causation (see
  `assumptions_and_limitations.md` for full discussion).
- A single change point model will miss additional regime shifts elsewhere in a
  35-year series.
- Nearest-event matching can mis-attribute a break to a coincidental rather than
  causally relevant event, particularly during periods with multiple overlapping
  events (e.g., 2008, 2020).


## 6. Summary Statistics

In [ ]:
print(f"Mean: ${df.Price.mean():.2f}")
print(f"Max:  ${df.Price.max():.2f} on {df.loc[df.Price.idxmax(), 'Date'].date()}")
print(f"Min:  ${df.Price.min():.2f} on {df.loc[df.Price.idxmin(), 'Date'].date()}")
print(f"Volatility (std log return): {df.log_return.std():.4f}")
